# Getting Started with Semantic Search

In this exercise, you will build a **semantic search engine for movies** using vector embeddings.

You will:
1. Prepare a movie dataset (titles + descriptions)
2. Convert movie descriptions into vector embeddings using a Hugging Face model
3. Store those vectors in a Qdrant vector database
4. Perform semantic search queries such as:

> *"I want to watch an adventure film about the Roman Empire"*

By the end, you'll understand how modern search systems go beyond keywords and use **meaning** instead.


## 1. Dataset Preparation

You are given a movie dataset from Kaggle.

For this exercise:
- Keep only the following columns:
  - `title`
  - `Description`
- Remove rows with missing descriptions
- Each movie description will later be converted into a vector embedding


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ishikajohari/imdb-data-with-descriptions")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
import pandas as pd

df = pd.read_csv(path + "/IMDB.csv")

df = df[["title", "Description"]].dropna().reset_index(drop=True)

print(f"Number of movies: {len(df)}")
df.head()


## 2. Install Required Libraries

We will use:
- `sentence-transformers` for embeddings
- `qdrant-client` to interact with Qdrant

In [ ]:
!pip install -q sentence-transformers qdrant-client

## 3. Setting Up Qdrant

Qdrant is a **vector database** optimized for semantic search.

You have two options:

### Option 1 (Recommended): Qdrant Cloud (Free Tier)
- Create a free account at https://cloud.qdrant.io
- Create a cluster
- Copy:
  - Cluster URL
  - API Key

### Option 2: Run Qdrant with Docker (Local)
```bash
docker run -p 6333:6333 qdrant/qdrant
```

## 4. Connect to Qdrant

Depending on your setup:
- Use an API key for Qdrant Cloud
- Or connect locally via `localhost:6333`

In [ ]:
from qdrant_client import QdrantClient

QDRANT_URL = "your-qdrant-endpoint"
QDRANT_API_KEY = "your-qdrant-key"

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
print(client.get_collections())


## 5. Embedding Movie Descriptions

We will use a Hugging Face embedding model to convert movie descriptions into vectors.

Model used:
- `sentence-transformers/all-MiniLM-L6-v2`

This model:
- Produces 384-dimensional embeddings
- Is fast and works well for semantic search



In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

EMBEDDING_DIM = model.get_sentence_embedding_dimension()
EMBEDDING_DIM

## 6. Create a Collection in Qdrant

A **collection** in Qdrant is like a table in a traditional database.

We will:
- Name the collection `movies`
- Store one vector per movie description
- Save the movie title as metadata (payload)

In [ ]:
from qdrant_client.models import VectorParams, Distance

COLLECTION_NAME = "movies"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE
    )
)

## 7. Upload Movie Vectors to Qdrant

Steps:
1. Convert each movie description into an embedding
2. Store:
   - Vector → description embedding
   - Payload → movie title + description

In [ ]:
from qdrant_client.models import PointStruct
from tqdm import tqdm

BATCH_SIZE = 100
points_batch = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    embedding = model.encode(row["Description"]).tolist()

    points_batch.append(
        PointStruct(
            id=idx,
            vector=embedding,
            payload={
                "title": row["title"],
                "description": row["Description"],
            },
        )
    )

    if len(points_batch) >= BATCH_SIZE:
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=points_batch,
        )
        points_batch.clear()

# upsert any remaining points
if points_batch:
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points_batch,
    )


## 8. Semantic Search

Now you can search using **natural language**, not keywords.

Example query:
> *"I want to watch an adventure film about the Roman Empire"*

Qdrant will:
- Embed the query
- Find the closest movie descriptions by meaning

In [ ]:
query = "I want to watch a movie about detectives and murders"

query_vector = model.encode(query).tolist()

hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
).points

for hit in hits:
    print(f"🎬 {hit.payload['title']}")
    print(hit.payload["description"])
    print("score:", hit.score)
    print("-" * 80)


## Bonus

Try the following:
1. Search for:
   - "A romantic movie set in Paris"
   - "A futuristic sci-fi movie with AI"
2. Change the distance metric to `DOT` or `EUCLID`
3. Add filters to your search (e.g. by year, genre or rating)
4. Batch embeddings instead of encoding one-by-one
5. Transform this notebook into a functional app (e.g. using Streamlit or Gradio)